# Gemma 4 26B — Few-Shot v5 (chat-format + structured output)

Diferencias frente a v4:

1. **Formato oficial de Gemma**: pasamos a `/api/chat` con rol `user`. Gemma instruction-tuned no tiene rol `system`, así que las instrucciones van dentro del mensaje de usuario; Ollama aplica automáticamente la chat template `<start_of_turn>user ... <end_of_turn><start_of_turn>model`.
2. **Structured output con JSON Schema**: en lugar de `format="json"` pasamos el schema completo como dict. Ollama (>=0.5) restringe la generación a la gramática del schema, eliminando los outputs basura tipo `pb_code_is_null`, `key_isrenamed_from_secondary_pbs` y JSON con `or null` mal interpretados que vimos en v4.
3. **Principio de decisión más preciso**: clima/temperatura/CO2 como **forcing experimental o escenario aplicado** activan el PB correspondiente (no sólo cuando se mide directamente). Esto cura el sesgo conservador del v4 que clasificaba como `None` papers donde el clima actúa como driver impuesto a un modelo downstream.
4. **Dos casos de calibración nuevos** (D y F) anclados en errores observados en runs anteriores.

In [ ]:
import pandas as pd
import requests
import json
import time
import os
from pathlib import Path

## 1. Carga de datos y reglas PB

In [ ]:
ROOT_DIR = Path.cwd().resolve().parents[2]
LLM_DIR = Path.cwd().resolve().parents[0]
OUTPUTS_DIR = LLM_DIR / 'outputs' / 'inferences'
GROUND_TRUTH_DIR = LLM_DIR / 'outputs' / 'ground_truth'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Los 3 doc_ids cuya narrativa se cita literalmente en los calibration cases A, B, C.
# Se excluyen del eval para que no haya filtración.
EXAMPLE_DOC_IDS = {'b9e1bf330baf', '9153f4dcf3d6', 'f21bc832abc6'}

df_corpus = pd.read_csv(ROOT_DIR / 'data' / 'corpus' / 'master_corpus_mixto_1000_clean_enriched.csv')
df_pbs = pd.read_csv(ROOT_DIR / 'corpus_PB' / 'data' / 'pb_reference.csv')
df_human = pd.read_csv(GROUND_TRUTH_DIR / 'validacion_real.csv', sep=';', encoding='utf-8')
ids_validados = df_human['doc_id'].dropna().astype(str).unique().tolist()

df_sample = df_corpus[df_corpus['doc_id'].isin(ids_validados)].copy()
df_sample = df_sample[~df_sample['doc_id'].isin(EXAMPLE_DOC_IDS)].copy()
print(f'Papers a evaluar (excluyendo los 3 ejemplos del prompt): {len(df_sample)}')

pb_rules = ''
for _, row in df_pbs.iterrows():
    pb_rules += f"- PB Code: {row['pb_code']} ({row['pb_name']})\n"
    pb_rules += f"  * Core Definition: {row['short_definition']}\n"
    pb_rules += f"  * UPV Context: Look for terms like: {row['applied_keywords_upv']}\n"
    pb_rules += f"  * ACTIVATION LOGIC: {row['activation_logic']}\n"
    pb_rules += f"  * EXCLUSION RULE (CRITICAL): {row['exclusion_notes']}\n\n"

PB_CODES = sorted(df_pbs['pb_code'].dropna().unique().tolist())
print('PB codes:', PB_CODES)

## 2. Prompt v5 — chat format + JSON schema enforcement

El schema fuerza:
- `primary_pb.pb_code` solo puede ser uno de `PB1..PB9` o `None` (literal). No más strings inventados.
- `confidence` solo puede ser `High|Medium|Low|N/A`.
- `rejected_pbs` solo puede contener códigos `PB1..PB9` (no se puede meter texto narrativo).

In [ ]:
OLLAMA_CHAT_URL = 'http://localhost:11434/api/chat'
OLLAMA_TAGS_URL = 'http://localhost:11434/api/tags'
MODEL_NAME = 'gemma4:26b'

PB_CODES_WITH_NONE = PB_CODES + ['None']

OUTPUT_SCHEMA = {
    'type': 'object',
    'properties': {
        'reasoning_process': {'type': 'string'},
        'primary_pb': {
            'type': 'object',
            'properties': {
                'pb_code': {'type': 'string', 'enum': PB_CODES_WITH_NONE},
                'confidence': {'type': 'string', 'enum': ['High', 'Medium', 'Low', 'N/A']},
            },
            'required': ['pb_code', 'confidence'],
        },
        'secondary_pbs': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'pb_code': {'type': 'string', 'enum': PB_CODES},
                    'confidence': {'type': 'string', 'enum': ['High', 'Medium', 'Low']},
                },
                'required': ['pb_code', 'confidence'],
            },
        },
        'rejected_pbs': {
            'type': 'array',
            'items': {'type': 'string', 'enum': PB_CODES},
        },
    },
    'required': ['reasoning_process', 'primary_pb', 'secondary_pbs', 'rejected_pbs'],
}


def preflight_check(model_name):
    """Aborta si Ollama no responde o el modelo no esta cargado."""
    try:
        r = requests.get(OLLAMA_TAGS_URL, timeout=5)
        r.raise_for_status()
    except Exception as e:
        raise RuntimeError(f'Ollama no responde en {OLLAMA_TAGS_URL}: {e}')
    available = [m.get('name') for m in r.json().get('models', [])]
    if model_name not in available:
        raise RuntimeError(
            f"Modelo '{model_name}' no cargado en Ollama. Disponibles: {available}. "
            f"Descargalo con: ollama pull {model_name}"
        )
    print(f"OK: Ollama responde y '{model_name}' disponible.")


# Plantilla del mensaje de usuario.
# IMPORTANTE: usamos un str.format simple en lugar de un f-string anidado para
# evitar el doble-interpolado del v4 ({High, Medium, Low} en el body causaba
# KeyError). Aqui los placeholders son solo {pb_rules} y {abstract_text}; las
# llaves literales ({{...}}) que aparezcan en el cuerpo (e.g. enums en texto)
# se escapan con {{ }} a la sintaxis de format.
USER_MESSAGE_TEMPLATE = """<expert_role>
You are a senior scientific evaluator analyzing research abstracts from the Universitat Politecnica de Valencia (UPV) against the Planetary Boundaries (PBs) framework. Your judgment matters more than rigid rule-following.
</expert_role>

<task>
Identify which Planetary Boundary the abstract actually MEASURES, MODELS, or treats as an EXPERIMENTAL FORCING. Cleanly separate active study from background motivation.
</task>

<decision_principle>
A PB is the PRIMARY boundary if any of these hold:
  (a) the paper measures or quantifies a variable from that PB's Core Definition;
  (b) the paper builds, runs or evaluates a model that produces values for that variable;
  (c) the paper applies that variable as an experimental treatment, imposed scenario or forcing whose downstream effects are then observed (e.g. a warming experiment, climate scenarios applied to a hydrological model, an elevated-CO2 ocean-acidification chamber, a UV-exposure assay).

A variable that ONLY appears in the motivation paragraph, with no quantification, modelling or imposition, is BACKGROUND and does NOT make its PB primary.

The verdict is "None" only when the abstract is purely social, legal, governance, educational, philosophical, or pure-software/methodological theory with no biophysical variable in any of roles (a)-(c).
</decision_principle>

<reference_framework>
{pb_rules}
</reference_framework>

<calibration_examples>
Use these to calibrate judgment. Do NOT copy their wording -- your abstract is different.

- Case A: a paper studies the central carbon metabolism of anammox bacteria using 13C isotope tracing. Reports specific metabolic pathways and quantified findings. Focus: nitrogen biogeochemistry (measured). Primary PB: PB4. Unambiguous.
- Case B: a paper verifies the WRF forecast model comparing initial conditions against radar. Mentions "changing climate" only in the opening. Focus: forecast-model accuracy. Climate is motivational backdrop, not measured/imposed. Primary PB: None.
- Case C: a paper quantifies detection data for 24 mammal species in response to human footprint. Focus: ecological response of biodiversity (measured with statistics). Primary PB: PB7. PB6 (land-system change) is contextual driver only.
- Case D: a paper proposes a systems-engineering framework for sustainable water-resources management; "rapid climate destabilization" appears in the motivation paragraph. Focus: water-resources planning and infrastructure performance. Climate is motivational; no climate variable is quantified or imposed. Primary PB: PB5. PB1 rejected.
- Case E: a multi-year warming experiment driving microbial functional gene expression in a grassland. Warming is the imposed forcing, microbes are the response. Climate is ACTIVE here (treatment). Primary PB: PB1. Secondary PB: PB7.
- Case F: a paper that propagates multiple climate-change scenarios through several hydrological models to quantify uncertainty in catchment-scale streamflow projections. Climate scenarios are IMPOSED on the hydrology model (case (c)); freshwater response is downstream. Primary PB: PB1. Secondary PB: PB5.
</calibration_examples>

<common_confusions>
- Aerosols / PM / AOD belong to PB9, not PB1.
- "Climate change", "global warming", "climate destabilization" appearing only in the motivation paragraph, with no climate variable measured / modelled / imposed, are NOT enough for PB1.
- Nutrients (N, P) and eutrophication point to PB4.
- Water quantity / quality / management point to PB5.
- Biodiversity / ecosystem responses are a legitimate primary candidate for PB7.
- Stratospheric ozone / UV-B exposure point to PB3.
</common_confusions>

<output_rules>
Return ONLY a JSON object that conforms to the schema enforced by the runtime.
- If no PB applies, set primary_pb.pb_code = "None" and primary_pb.confidence = "N/A".
- Otherwise pick exactly one of PB1-PB9 as primary, with confidence in {{High, Medium, Low}}.
- secondary_pbs and rejected_pbs may be empty arrays. Codes must be in PB1-PB9.
- reasoning_process: 2-3 sentences. State what is measured/modelled/imposed, why this PB, why it is not background. Cite the abstract's own terms; do not echo this prompt.
</output_rules>

<abstract>
{abstract_text}
</abstract>"""


def build_user_message(abstract_text):
    """Mensaje de usuario unico (Gemma no tiene rol system: las instrucciones van aqui)."""
    return USER_MESSAGE_TEMPLATE.format(pb_rules=pb_rules, abstract_text=abstract_text)


def classify_abstract_v5(abstract_text):
    payload = {
        'model': MODEL_NAME,
        'messages': [
            {'role': 'user', 'content': build_user_message(abstract_text)},
        ],
        'stream': False,
        'format': OUTPUT_SCHEMA,
        'options': {
            'temperature': 0.0,
            'top_p': 0.95,
            'top_k': 64,
            'num_ctx': 8192,
        },
        'keep_alive': '15m',
    }
    t0 = time.time()
    try:
        r = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=300)
        r.raise_for_status()
        data = r.json()
        content = data.get('message', {}).get('content', '{}')
        return content, time.time() - t0, None
    except Exception as e:
        err = f'{type(e).__name__}: {str(e)[:200]}'
        return json.dumps({'error': err}), time.time() - t0, err

## 3. Parser (más simple porque el schema ya garantiza la estructura)

In [ ]:
def parse_llm_output(raw_text):
    try:
        si, ei = raw_text.find('{'), raw_text.rfind('}')
        if si == -1 or ei == -1:
            raise ValueError('No JSON object in response')
        data = json.loads(raw_text[si:ei + 1])
        reasoning = data.get('reasoning_process', '') or ''
        prim = data.get('primary_pb') or {}
        prim_code = prim.get('pb_code', 'None') if isinstance(prim, dict) else 'None'
        prim_conf = prim.get('confidence', 'N/A') if isinstance(prim, dict) else 'N/A'
        sec = data.get('secondary_pbs') or []
        sec_codes = ', '.join(s.get('pb_code', '') for s in sec if isinstance(s, dict)) or 'None'
        rej = data.get('rejected_pbs') or []
        rej_codes = ', '.join(str(x) for x in rej) if rej else 'None'
        return reasoning, prim_code, prim_conf, sec_codes, rej_codes
    except Exception as e:
        return f'Error: {e}', 'Error', 'Error', 'Error', 'Error'

## 4. Smoke test — los 2 papers que no convencieron en v4

Verifica que v5 (a) ya no produce tokens basura en `rejected_pbs` y (b) clasifica correctamente el paper de hidrología+escenarios climáticos.

Esperado:
- `b39624d6c38a` (hydrological models forced with climate scenarios): GT = PB1 + PB5. v4 dijo None. v5 debería decir PB1 (Primary) + PB5 (secondary).
- `8581e74341ad` (PM2.5/AOD aerosol satellite paper): v4 devolvió `pb_code_is_null` y otros tokens basura aunque su propio reasoning concluía PB9. Con structured output debe salir PB9 limpio.

In [ ]:
preflight_check(MODEL_NAME)

smoke_ids = ['b39624d6c38a', '8581e74341ad']
df_smoke = df_corpus[df_corpus['doc_id'].isin(smoke_ids)].copy()
for _, row in df_smoke.iterrows():
    print(f"\n=== {row['doc_id']} — {row['title'][:80]}")
    raw, t, err = classify_abstract_v5(row['clean_abstract'])
    if err:
        print(f'FALLO ({t:.1f}s): {err}')
        continue
    reasoning, p, pc, s, r = parse_llm_output(raw)
    print(f'[{t:.1f}s] Pri: {p}/{pc} | Sec: {s} | Rej: {r}')
    print(f'Reason: {reasoning[:300]}')

## 5. Bucle de evaluación completo

In [ ]:
output_filename = OUTPUTS_DIR / 'gemma4_26b_fewshot_v5_principle.csv'
if output_filename.exists():
    output_filename.unlink()

preflight_check(MODEL_NAME)

total = len(df_sample)
print(f'FEWSHOT v5 (chat + structured output) | modelo: {MODEL_NAME}')
print(f'Excluidos del eval: {sorted(EXAMPLE_DOC_IDS)}')
print('-' * 70)

err_count = 0
for i, (_, row) in enumerate(df_sample.iterrows(), start=1):
    print(f"[{i}/{total}] {row['doc_id']}...", end=' ', flush=True)
    raw, t, err = classify_abstract_v5(row['clean_abstract'])
    if err:
        err_count += 1
        print(f'[{t:.2f}s] FALLO -> {err}')
        if err_count >= 3:
            print('\n>>> 3 fallos consecutivos, abortando para que diagnostiques. <<<')
            break
    else:
        err_count = 0
    reasoning, p, pc, s, r = parse_llm_output(raw)
    if not err:
        print(f'[{t:.1f}s] Pri: {p}/{pc} | Sec: {s} | Rej: {r}')
    pd.DataFrame([{
        'doc_id': row['doc_id'],
        'llm_primary_pb': p,
        'llm_primary_conf': pc,
        'llm_secondary_pbs': s,
        'llm_rejected_pbs': r,
        'llm_reasoning': reasoning,
        'inference_time_sec': t,
    }]).to_csv(output_filename, mode='a', header=not output_filename.exists(), index=False)

print('=' * 70)
print(f'OK. CSV: {output_filename}')